In [1]:
!pip install -q torch torchvision wandb

In [2]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [3]:
import wandb
wandb.login(WANDB_API_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sagar-sharma-03015611924 (practicum-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# BASE TRAINING AND EVALUATION

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import time
import tracemalloc
import os

In [ ]:
EPOCHS = 150 # How many times the model sees the dataset
BATCH_SIZE = 128 # How many samples per gradient update
LR = 0.1 # (learning rate) Controls step size in weight updates
WEIGHT_DECAY = 5e-4 # Penalizes large weights (l2 regularization)
MOMENTUM = 0.9 #Adds “inertia” to updates. Instead of reacting only to current gradient, it remembers past direction.
NUM_WORKERS = 2 # How many CPU processes load data
SAVE_PATH = "resnet18_cifar10_baseline.pth" # Where model weights are stored
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Where computation happens
LATENCY_RUNS = 100 # Number of times you measure inference speed
WARMUP_RUNS = 10 # Initial runs before measuring latency

 1. Start with standard ResNet-18 architecture (originally designed for ImageNet: 224x224 image

 2. Modify first convolution:
    - Original: 7x7 kernel, stride=2 (aggressive downsampling)
    - CIFAR-10 images are only 32x32 → early downsampling destroys spatial detail
    - Use 3x3, stride=1 to preserve more information.

3. Remove maxpool layer:
    - MaxPool would further halve spatial dimensions
    - For small images, this leads to excessive information loss early in the network
    - Identity() keeps the data unchanged

4.  Adjust final fully connected layer:
    - Original ResNet outputs 1000 classes (ImageNet)
    - CIFAR-10 has only 10 classes → match output dimension
    - 512 = feature size from ResNet backbone (must match exactly)


In [ ]:
def get_cifar_resnet18(num_classes=10):
    model = resnet18(weights=None)
    # Modify first convolution:
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Remove maxpool layer
    model.maxpool = nn.Identity()
    # Adjust final fully connected layer
    model.fc = nn.Linear(512, num_classes)
    return model

1. Data augmentation for training:
    - RandomCrop with padding: simulates small translations (object shifts)
    - RandomHorizontalFlip: makes model invariant to left-right orientation
    - ToTensor: converts image to tensor and scales pixels to [0,1]
    - Normalize: standardizes input using CIFAR-10 mean & std for stable training

2. No augmentation for test data:
    - We want consistent, real evaluation (not random variations)
    - Only convert to tensor and normalize
    
3. Load CIFAR-10 training dataset with augmentation

4. Load CIFAR-10 test dataset without augmentation

5. DataLoader for training:
    - shuffle=True: ensures batches are randomly mixed → better gradient updates
    - num_workers: parallel data loading for speed

6. DataLoader for testing:
    - shuffle=False: keeps evaluation deterministic and reproducible

In [ ]:
def get_loaders():
    # Data augmentation for training
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])
    # No augmentation for test data
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    # Load CIFAR-10 training dataset with augmentation
    train_set = torchvision.datasets.CIFAR10(root="./data", train=True,download=True, transform=train_transform)

    # Load CIFAR-10 test dataset without augmentation
    test_set  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

    # DataLoader for training
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,shuffle=True, num_workers=NUM_WORKERS)

    # DataLoader for testing
    test_loader  = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,shuffle=False, num_workers=NUM_WORKERS)

    return train_loader, test_loader


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    # Set model to training mode
    model.train()

    total_loss, correct, total = 0,0,0

    for images, labels in loader:
        # Move data to GPU/CPU
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        # Clear old gradients (otherwise they accumulate)
        optimizer.zero_grad()
        # Forward pass: get predictions
        outputs = model(images)
        # Compute loss between predictions and true labels
        loss = criterion(outputs, labels)
        # Backpropagation: compute gradients
        loss.backward()
        # Update model weights
        optimizer.step()
        # Accumulate total loss (scaled by batch size)
        total_loss += loss.item() * labels.size(0)
        # Get predicted class (index of max logit)
        _, predicted = outputs.max(1)
        # Count correct predictions
        correct += predicted.eq(labels).sum().item()
        # Total samples processed
        total += labels.size(0)

    return total_loss / total, 100.0 *  correct / total

def evaluate_accuracy(model, loader):
    # Set model to evaluation mode
    model.eval()

    correct, total = 0, 0

    # Disable gradient computation (saves memory + faster)
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return 100.0 * correct / total # accuracy in percentage


Benchmarking (CPU)

In [ ]:
def measure_latency(model):
    model.eval().to("cpu")
    # Dummy input matching CIFAR-10 shape (batch=1)
    dummy = torch.randn(1,3,32,32)

    with torch.no_grad():
        # Warmup runs
        for _ in range(WARMUP_RUNS):
            _ = model(dummy)
        times = []

        # latency measurement
        for _ in range(LATENCY_RUNS):
            start = time.perf_counter()
            _ = model(dummy)
            times.append((time.perf_counter()-start) * 1000) # convert to ms

    return sum(times)/len(times)

def measure_ram(model):
    model.eval().to("cpu")
    dummy = torch.randn(1, 3, 32, 32)

    tracemalloc.start()

    with torch.no_grad():
        _ = model(dummy)
        _, peak = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return peak / (1024 ** 2) # convert bytes to MB

def get_model_size_mb(model):
    torch.save(model.state_dict(), "_tmp.pth")
    size = os.path.getsize("_tmp.pth") / (1024 ** 2)
    os.remove("_tmp.pth")

    return size

def count_params(model):
    # Count total number of trainable parameters
    return sum(p.numel() for p in model.parameters())

In [ ]:
EPOCHS = 150

def run_train_test():
    # Initialize Weights & Biases experiment tracking
    # Logs hyperparameters, metrics, and allows experiment comparison
    wandb.init(
        project="cnn-compression",
        name="baseline_150epoch",
        id="baseline_150epoch",
        resume="allow",
        config={
            "model": "resnet18",
            "dataset": "cifar10",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "momentum": MOMENTUM,
            "scheduler": "cosine_annealing",
            "optimizer": "sgd",
            "device": str(DEVICE),
            "latency_runs": LATENCY_RUNS,
            "warmup_runs": WARMUP_RUNS
    })
    # Load training and test data
    train_loader, test_loader = get_loaders()
    # Initialize model and move to device (GPU/CPU)
    model = get_cifar_resnet18().to(DEVICE)
    # Loss function for multi-class classification
    criterion = nn.CrossEntropyLoss()
    # Optimizer: SGD with momentum + weight decay (regularization)
    optimizer = optim.SGD(
        model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    # Learning rate scheduler:
    # Cosine annealing gradually reduces LR over epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )
    best_acc = 0.0  # Track best test accuracy
    # Training loop
    for epoch in range(1, EPOCHS + 1):
        # Train for one epoch
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion
        )
        # Evaluate on test set
        test_acc = evaluate_accuracy(model, test_loader)
        # Update learning rate
        scheduler.step()
        # Print progress
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Loss: {train_loss:.4f} | "
              f"Train: {train_acc:.2f}% | "
              f"Test: {test_acc:.2f}%")
        # Log metrics to wandb
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_acc": test_acc,
            "lr": scheduler.get_last_lr()[0],
        }, step=epoch)
        # Save best model based on test accuracy
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"  ✓ Saved best model ({best_acc:.2f}%)")

    # Evaluation of best model
    # Load best saved model (on CPU for consistent evaluation)
    model.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
    model = model.to("cpu")
    # Compute final test accuracy (fresh loader for safety)
    final_acc = evaluate_accuracy(
        model.to("cpu"),
        torch.utils.data.DataLoader(
            torchvision.datasets.CIFAR10(
                root="./data",
                train=False,
                transform=transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize(
                        (0.4914, 0.4822, 0.4465),
                        (0.2023, 0.1994, 0.2010)
                    )
                ])
            ),
            batch_size=BATCH_SIZE,
            shuffle=False
        )
    )
    # Efficiency metrics
    latency  = measure_latency(model)     # inference speed
    ram      = measure_ram(model)         # peak RAM usage
    size_mb  = get_model_size_mb(model)   # model file size
    n_params = count_params(model)        # number of parameters
    # Print final results
    print(f"  Top-1 Accuracy  : {final_acc:.2f}%")
    print(f"  Model Size      : {size_mb:.2f} MB")
    print(f"  Parameters      : {n_params:,}")
    print(f"  Latency (avg)   : {latency:.3f} ms")
    print(f"  Peak RAM        : {ram:.2f} MB")
    # Log final metrics
    wandb.log({
        "sparsity": 0.0,
        "top1_accuracy": final_acc,
        "model_size_mb": size_mb,
        "latency_ms": latency,
        "peak_ram_mb": ram,
        "nonzero_params": n_params,
    })

    wandb.save(SAVE_PATH)

In [ ]:
def evaluate_model_cpu():
    model = get_cifar_resnet18()
    model.load_state_dict(torch.load(SAVE_PATH, map_location="cpu"))
    model.eval()

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])
    cpu_test_loader = torch.utils.data.DataLoader(
        torchvision.datasets.CIFAR10(root="./data", train=False,
                                     transform=test_transform),
        batch_size=BATCH_SIZE, shuffle=False
    )

    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in cpu_test_loader:
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    final_acc = 100.0 * correct / total

    latency  = measure_latency(model)
    ram      = measure_ram(model)
    size_mb  = get_model_size_mb(model)
    n_params = count_params(model)

    print(f"  Top-1 Accuracy  : {final_acc:.2f}%")
    print(f"  Model Size      : {size_mb:.2f} MB")
    print(f"  Parameters      : {n_params:,}")
    print(f"  Latency (avg)   : {latency:.3f} ms")
    print(f"  Peak RAM        : {ram:.2f} MB")

    wandb.log({
        "sparsity": 0.0,
        "top1_accuracy": final_acc,
        "model_size_mb": size_mb,
        "latency_ms": latency,
        "peak_ram_mb": ram,
        "nonzero_params": n_params,
    })

In [ ]:
run_train_test()

100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s]


Epoch   1/150 | Loss: 2.0493 | Train: 26.03% | Test: 39.84%
  ✓ Saved best model (39.84%)
Epoch   2/150 | Loss: 1.4476 | Train: 46.64% | Test: 53.55%
  ✓ Saved best model (53.55%)
Epoch   3/150 | Loss: 1.1249 | Train: 59.81% | Test: 63.90%
  ✓ Saved best model (63.90%)
Epoch   4/150 | Loss: 0.9119 | Train: 67.69% | Test: 67.60%
  ✓ Saved best model (67.60%)
Epoch   5/150 | Loss: 0.7544 | Train: 73.71% | Test: 69.03%
  ✓ Saved best model (69.03%)
Epoch   6/150 | Loss: 0.6493 | Train: 77.33% | Test: 78.96%
  ✓ Saved best model (78.96%)
Epoch   7/150 | Loss: 0.5929 | Train: 79.61% | Test: 73.21%
Epoch   8/150 | Loss: 0.5462 | Train: 81.19% | Test: 78.32%
Epoch   9/150 | Loss: 0.5166 | Train: 82.10% | Test: 66.85%
Epoch  10/150 | Loss: 0.4882 | Train: 83.15% | Test: 74.49%
Epoch  11/150 | Loss: 0.4784 | Train: 83.56% | Test: 77.40%
Epoch  12/150 | Loss: 0.4612 | Train: 84.12% | Test: 78.74%
Epoch  13/150 | Loss: 0.4443 | Train: 84.90% | Test: 78.33%
Epoch  14/150 | Loss: 0.4374 | Train: 85

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
evaluate_model_cpu()

In [ ]:
wandb.finish()

# PRUNING